# Exercise 8: Put all the concepts in Exercise 7 together

Skills:
* Apply all the concepts covered in Exercise 7 for a research question. Know when to use what concept.

References: 
* Exercise 7


### To Do

Narrow down the list of rail routes in CA to 3 groups. Use the SHN network to determine how much of the rail route runs near the SHN. We care only about rail routes that run entirely in CA (use stops to figure this out).

**Near** the interstate, US highway, or state highway is defined by being within a quarter mile. For this exercise, the distinction between interstate, US highway, and state highway is not important; treat any road that shows up in the dataset as "the SHN".

There are theoretically 3 groupings: 
* rail routes that are never within 0.25 miles of the SHN
* rail routes with > 0 but less than half of its length near the SHN 
* rail routes with at least half of its length near the SHN

Provide a table and a chart showing how many rail routes fall into each of the 3 groups by district.

Use a Markdown cell at the end to connect which geospatial concept was applied to which step of the process. The concepts that should be used are `projecting CRS`, `buffering`, `dissolve`, `clipping`, `spatial join`, `overlay`. 

In [1]:
import geopandas as gpd
import intake
import pandas as pd

catalog = intake.open_catalog("shared_data_catalog.yml") # Note: need to change catalog path

In [2]:
# Import data
districts = catalog.caltrans_districts.read()
highways = catalog.state_highway_network.read()

rail_group = ['0', '1', '2'] # For some reason this filters out San Joaquins routes & stops and zephyr routes only - not sure why
routes = catalog.ca_transit_routes.read()
rail_routes = routes.loc[
    routes.route_type.isin(rail_group)
].reset_index(drop=True)

stops = catalog.ca_transit_stops.read()

In [3]:
# Filter for rail stops
stops_route_type_processing = stops.copy()
# Get routetypes as an iterable so it can be exploded
stops_route_type_processing["routetypes_iterable"] = stops_route_type_processing["routetypes"].str.split(", ")
# Exploded the stops df on routetypes
stops_exploded_by_route_type = stops_route_type_processing.explode(
    "routetypes_iterable"
).rename(
    columns={"routetypes_iterable": "route_type"}
).reset_index(names="original_index") # Keep the original index so we can identify duplicate stops later
rail_stops = stops_exploded_by_route_type.loc[
    stops_exploded_by_route_type.route_type.isin(rail_group)
].drop_duplicates( # Drop any duplicate stops after we filter by route type, in case they had multiple of the listed route types
    subset="original_index"
).reset_index(
    drop=True
).drop( # Drop original_index, since we don't need it anymore
    "original_index", axis=1
)


In [4]:
rail_routes.head()

,org_id,agency,route_id,route_type,route_name,shape_id,n_trips,base64_url,geometry
0,recUmm4gcNXaqrwpn,Sonoma-Marin Area Rail Transit District,SMART,2,Sonoma County Airport to Larkspur,p_897679,21,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,"LINESTRING (-122.78419 38.50989, -122.78357 38..."
1,recUmm4gcNXaqrwpn,Sonoma-Marin Area Rail Transit District,SMART,2,Sonoma County Airport to Larkspur,p_897680,21,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,"LINESTRING (-122.51244 37.94780, -122.51233 37..."
2,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,801,0,Metro A Line,801NB_RC_221121,122,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (-118.19297 33.76800, -118.19339 33..."
3,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,801,0,Metro A Line,801SB_RC_221121,124,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (-117.89010 34.13701, -117.89044 34..."
4,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,802,1,Metro B Line,802EB_190513,89,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (-118.37827 34.17080, -118.37659 34..."


In [5]:
rail_stops.head()

,org_id,agency,stop_id,stop_name,n_routes,route_ids_served,routetypes,n_arrivals,n_hours_in_service,meters_to_ca_state_highway,base64_url,district_name,geometry,route_type
0,recC5CT95EufmQCXr,Santa Clara Valley Transportation Authority,64811,Reamwood Station,1,Orange Line,0,62,18,808.1,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,04 - Bay Area / Oakland,POINT (-121.98966 37.40357),0
1,recC5CT95EufmQCXr,Santa Clara Valley Transportation Authority,64811,Reamwood Station,1,Orange Line,0,62,18,808.1,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,04 - Bay Area / Oakland,POINT (-121.98966 37.40357),0
2,recC5CT95EufmQCXr,Santa Clara Valley Transportation Authority,64812,Vienna Station,1,Orange Line,0,62,18,567.0,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,04 - Bay Area / Oakland,POINT (-122.00087 37.40340),0
3,recC5CT95EufmQCXr,Santa Clara Valley Transportation Authority,64812,Vienna Station,1,Orange Line,0,62,18,567.0,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,04 - Bay Area / Oakland,POINT (-122.00087 37.40340),0
4,recC5CT95EufmQCXr,Santa Clara Valley Transportation Authority,64813,Fair Oaks Station,1,Orange Line,0,62,18,422.9,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,04 - Bay Area / Oakland,POINT (-122.00955 37.40278),0


In [6]:
# Filter to get stops not in CA only
caltrans_districts = catalog.caltrans_districts.read()
ca = caltrans_districts.dissolve()
stops_not_in_ca = rail_stops.copy().overlay(ca, how="difference") # .copy() here avoids annoying-to-trace SettingWithCopyWarning
stops_not_in_ca["route_ids_as_iterable"] = stops_not_in_ca["route_ids_served"].str.split(", ")
stops_route_exploded_not_in_ca = stops_not_in_ca.explode(
    "route_ids_as_iterable"
).rename(
    columns={"route_ids_as_iterable": "route_id"}
).drop(
    "route_ids_served", axis=1
).reset_index(
    drop=True
)
# Get the route ids for routes that stop outside of CA
non_ca_route_ids = stops_route_exploded_not_in_ca[["org_id", "route_id"]].drop_duplicates()
# Filter routes that stop outside of CA
rail_routes_in_ca = rail_routes.loc[
    ~(rail_routes["org_id"].isin(non_ca_route_ids["org_id"]) & rail_routes["route_id"].isin(non_ca_route_ids["route_id"]))
].reset_index(drop=True).to_crs("EPSG:3310")
ones = pd.Series(1, index=rail_routes_in_ca.index)
rail_routes_in_ca["unique_route_id"] = ones.cumsum()
rail_routes_in_ca["length"] = rail_routes_in_ca.length

In [7]:
METER_TO_MILE_CONVERSION = 0.000621371
# Get the ca SHN from the catalog
shn = catalog.state_highway_network.read().to_crs("EPSG:3310")
# Buffer it by 0.25 miles
shn["buffer_geometry"] = shn.buffer(0.25 / METER_TO_MILE_CONVERSION)

In [8]:
rail_near_shn = rail_routes_in_ca.overlay(
    shn.set_geometry("buffer_geometry").dissolve(), how="intersection"
).dissolve(
    "unique_route_id"
)[[rail_routes_in_ca.geometry.name]]
rail_near_shn["length_near_shn"] = rail_near_shn.length

In [9]:
from enum import Enum
import numpy as np

NOT_NEAR_SHN = "Not Near SHN"
MINORITY_NEAR_SHN = "Minority Near SHN"
MAJORITY_NEAR_SHN = "Majority Near SHN"

def categorize_shn_proximity(shn_proximity: float) -> str:
    if shn_proximity == 0:
        return NOT_NEAR_SHN
    if shn_proximity < 0.5:
        return MINORITY_NEAR_SHN
    if shn_proximity >= 0.5:
        return MAJORITY_NEAR_SHN
    return np.nan

rail_shn_merged = rail_routes_in_ca.merge(
    rail_near_shn["length_near_shn"],
    how="left",
    left_on="unique_route_id",
    right_index=True,
    validate="one_to_one"
)
rail_shn_merged["length_near_shn"] = rail_shn_merged["length_near_shn"].fillna(0)
rail_shn_merged["proportion_near_shn"] = rail_shn_merged["length_near_shn"] / rail_shn_merged["length"]
rail_shn_merged["category"] = rail_shn_merged["proportion_near_shn"].map(categorize_shn_proximity)
rail_shn_merged.head(10)

,org_id,agency,route_id,route_type,route_name,shape_id,n_trips,base64_url,geometry,unique_route_id,length,length_near_shn,proportion_near_shn,category
0,recUmm4gcNXaqrwpn,Sonoma-Marin Area Rail Transit District,SMART,2,Sonoma County Airport to Larkspur,p_897679,21,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,"LINESTRING (-242468.236 58422.620, -242417.053...",1,72274.308522,30004.709324,0.415150,Minority Near SHN
1,recUmm4gcNXaqrwpn,Sonoma-Marin Area Rail Transit District,SMART,2,Sonoma County Airport to Larkspur,p_897680,21,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,"LINESTRING (-220464.313 -4701.385, -220453.638...",2,72269.153862,30032.996833,0.415571,Minority Near SHN
2,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,801,0,Metro A Line,801NB_RC_221121,122,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (167422.001 -470302.482, 167383.547...",3,78082.188759,40896.047631,0.523756,Majority Near SHN
3,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,801,0,Metro A Line,801SB_RC_221121,124,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (194568.172 -428802.944, 194537.371...",4,78062.392014,40977.915958,0.524938,Majority Near SHN
4,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,802,1,Metro B Line,802EB_190513,89,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (149491.917 -425941.024, 149652.155...",5,25198.648117,7608.875122,0.301956,Minority Near SHN
5,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,802,1,Metro B Line,802WB_190513,89,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (163224.448 -439551.137, 163223.318...",6,25211.774750,7648.078544,0.303353,Minority Near SHN
6,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,803,0,Metro C Line,803NB_241015,97,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (175311.870 -453943.902, 175306.430...",7,28533.318360,26819.655220,0.939942,Majority Near SHN
7,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,803,0,Metro C Line,803SB_241015,99,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (149866.474 -450434.525, 149867.055...",8,28534.057747,26831.059071,0.940317,Majority Near SHN
8,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,804,0,Metro E Line,804EB_RC_221121,123,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (139329.057 -443544.110, 139344.251...",9,35450.645726,17595.941949,0.496350,Minority Near SHN
9,recPnGkwdpnr8jmHB,Los Angeles County Metropolitan Transportation...,804,0,Metro E Line,804WB_RC_221121,120,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX3JhaW...,"LINESTRING (170545.635 -440800.541, 170457.977...",10,35457.125406,17417.560553,0.491229,Minority Near SHN


In [10]:
rail_shn_district_merged = rail_shn_merged.sjoin(
    caltrans_districts.to_crs("EPSG:3310")[[caltrans_districts.geometry.name, "DISTRICT"]], how="left", predicate="intersects"
)
district_summary_table = rail_shn_district_merged.groupby("DISTRICT")["category"].value_counts().rename(index="n_projects").reset_index()

In [11]:
district_summary_table

,DISTRICT,category,n_projects
0,3,Minority Near SHN,45
1,3,Not Near SHN,9
2,4,Minority Near SHN,219
3,4,Majority Near SHN,67
4,5,Minority Near SHN,12
5,7,Minority Near SHN,26
6,7,Majority Near SHN,4
7,10,Minority Near SHN,2
8,11,Minority Near SHN,58
9,11,Majority Near SHN,56


In [18]:
import altair as alt
alt.Chart(district_summary_table, title="District Summary").mark_bar(width=20).encode(
    x=alt.X("DISTRICT", title="District"),
    y=alt.Y("n_projects", title="# Projects"),
    color="category"
)

alt.Chart(...)